To run this code, first download the 311 requests and zip code datasets: <br />
Austin 311 requests: https://catalog.data.gov/dataset/311-service-requests-austin-transportation-and-public-works
Austin Zip Code Boundaries: https://catalog.data.gov/dataset/boundaries-us-zip-codes

In [ ]:
! pip install geopandas shapely

In [ ]:
import pandas as pd
import geopandas as gpd
from shapely import wkt

In [ ]:
# ============================================================
# 1. FILE PATHS
# ============================================================

# Make sure these match the filepaths to your files
REQUESTS_FILE = "datasets/austin_311_requests.csv"
ZIP_FILE = "datasets/austin_zip_codes.csv"


# ============================================================
# 2. LOAD DATA
# ============================================================

requests = pd.read_csv(REQUESTS_FILE)
zips = pd.read_csv(ZIP_FILE)


# ============================================================
# 3. PREPARE ZIP CODE POLYGONS
# ============================================================

# Convert the WKT MULTIPOLYGON text into actual geometry
zips["geometry"] = zips["the_geom"].apply(wkt.loads)

zip_gdf = gpd.GeoDataFrame(
    zips,
    geometry="geometry",
    crs="EPSG:4326"
)

# Keep only the fields we actually need
zip_gdf = zip_gdf[["ZIPCODE", "geometry"]]


# ============================================================
# 4. PREPARE 311 LOCATIONS
# ============================================================

# Convert POINT (...) text into geometry
requests_geo = requests.dropna(subset=["Location"]).copy()

requests_geo["geometry"] = requests_geo["Location"].apply(wkt.loads)

request_gdf = gpd.GeoDataFrame(
    requests_geo,
    geometry="geometry",
    crs="EPSG:4326"
)


# ============================================================
# 5. MATCH 311 REQUESTS TO ZIP CODES
# ============================================================

request_gdf = gpd.sjoin(
    request_gdf,
    zip_gdf,
    how="left",
    predicate="within"
)

request_gdf = request_gdf.rename(
    columns={"ZIPCODE": "ZIP_CODE"}
)

request_gdf = request_gdf.drop(
    columns=["geometry", "index_right"],
    errors="ignore"
)


# ============================================================
# 6. EXPORT POWER BI-READY DATA
# ============================================================

request_gdf.to_csv(
    "311_requests_with_zip.csv",
    index=False
)

print("Done!")
print("Created:")
print("  - 311_requests_with_zip.csv")